# STEP 1: Import

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from delta.tables import DeltaTable

#### **Insights**

##### Imports all required PySpark and Delta Lake libraries for data processing.

# STEP 2: Create SparkSession

In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
        .appName("Delta_Lake") \
        .getOrCreate()

#### **Insights**

##### Initializes a Spark session to execute Spark.

# STEP 3: Upload Dataset

In [0]:
df = spark.read.format("csv")\
    .option("header",True)\
    .option("inferSchema",True)\
    .option("mode","PERMISSIVE")\
    .load("/Volumes/dbacademy/default/myvolume/employee_management_20000.csv")

display(df.limit(10))

employee_id,employee_name,gender,age,department,designation,salary,experience,joining_date,city,state,manager_name,performance_rating,attendance_percentage,active
1,Employee_1,Male,42,Finance,Engineer,45000,5,02/01/2023,Pune,Gujarat,Neha,6,110,true
2,Employee_2,Female,-3,IT,Analyst,25000,-1,01-03-2023,Pune,Gujarat,null,5,97,false
3,Employee_3,Male,24,IT,Manager,45000,5,2023-01-04,Ahmedabad,Rajasthan,Rohit,5,88,false
4,Employee_4,Male,36,Sales,Manager,-15000,1,05/01/2023,Ahmedabad,Delhi,Rohit,3,88,false
5,Employee_5,Female,-3,Finance,Executive,70000,5,01-06-2023,Pune,Maharashtra,Amit,6,88,true
6,Employee_6,Male,36,Sales,Analyst,70000,1,2023-01-07,Pune,Gujarat,null,4,92,true
7,Employee_7,Male,30,Sales,Executive,25000,5,08/01/2023,Mumbai,Delhi,Rohit,6,110,false
8,Employee_8,Female,36,Marketing,Executive,45000,3,01-09-2023,Pune,Rajasthan,Rohit,6,110,false
9,Employee_9,Female,36,Sales,Executive,-15000,5,2023-01-10,Jaipur,Gujarat,Rohit,6,92,true
10,Employee_10,Female,36,Sales,Engineer,25000,5,11/01/2023,Jaipur,Rajasthan,null,6,92,false


#### **Insights**

##### Loads the employee CSV dataset into a Spark DataFrame.

# STEP 4: Check Data

In [0]:
print("Showing only 5 records")
display(df.limit(5))

Showing only 5 records


employee_id,employee_name,gender,age,department,designation,salary,experience,joining_date,city,state,manager_name,performance_rating,attendance_percentage,active
1,Employee_1,Male,42,Finance,Engineer,45000,5,02/01/2023,Pune,Gujarat,Neha,6,110,true
2,Employee_2,Female,-3,IT,Analyst,25000,-1,01-03-2023,Pune,Gujarat,null,5,97,false
3,Employee_3,Male,24,IT,Manager,45000,5,2023-01-04,Ahmedabad,Rajasthan,Rohit,5,88,false
4,Employee_4,Male,36,Sales,Manager,-15000,1,05/01/2023,Ahmedabad,Delhi,Rohit,3,88,false
5,Employee_5,Female,-3,Finance,Executive,70000,5,01-06-2023,Pune,Maharashtra,Amit,6,88,true


#### **Insights**

##### Displays the dataset for quick verification.

In [0]:
print("The Schema is: ")
df.printSchema()

The Schema is: 
root
 |-- employee_id: integer (nullable = true)
 |-- employee_name: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- department: string (nullable = true)
 |-- designation: string (nullable = true)
 |-- salary: integer (nullable = true)
 |-- experience: integer (nullable = true)
 |-- joining_date: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- manager_name: string (nullable = true)
 |-- performance_rating: integer (nullable = true)
 |-- attendance_percentage: integer (nullable = true)
 |-- active: boolean (nullable = true)



#### **Insights**

##### Confirms the data types and structure of the dataset.

In [0]:
print("Number of rows in the dataset are : ",df.count())

print("Number of columns in the dataset are : ",len(df.columns))

Number of rows in the dataset are :  1332
Number of columns in the dataset are :  15


#### **Insights**

##### Provides the dataset size before data cleaning.

In [0]:
print("The columns in the dataset are : ")
df.columns

The columns in the dataset are : 


['employee_id',
 'employee_name',
 'gender',
 'age',
 'department',
 'designation',
 'salary',
 'experience',
 'joining_date',
 'city',
 'state',
 'manager_name',
 'performance_rating',
 'attendance_percentage',
 'active']

#### **Insights**

##### Lists columns for reference.

# Step 5: Data Cleaning

##5.1:  Checking Null Values

In [0]:
null_values = df.select([
    count(when(col(x).isNull(), x)).alias(x)
    for x in df.columns
])

display(null_values)

employee_id,employee_name,gender,age,department,designation,salary,experience,joining_date,city,state,manager_name,performance_rating,attendance_percentage,active
0,63,0,0,0,0,0,0,0,0,0,330,74,1,1


#### **Insights**

##### Identifies missing values in each column.

##5.2: Dropping Null Values

In [0]:
df.count()

1332

In [0]:
cleaned_df = df.dropna(subset = ['employee_name'])

In [0]:
cleaned_df.count()

1269

#### **Insights**

##### Removes records where employee names are missing.

##5.3: Filling Null Values

In [0]:
cleaned_df = cleaned_df.fillna({
    "manager_name": "Not Assigned",
    "performance_rating": 0,
    "attendance_percentage":0,
    "active": False
})

In [0]:
display(cleaned_df.limit(10))

employee_id,employee_name,gender,age,department,designation,salary,experience,joining_date,city,state,manager_name,performance_rating,attendance_percentage,active
1,Employee_1,Male,42,Finance,Engineer,45000,5,02/01/2023,Pune,Gujarat,Neha,6,110,true
2,Employee_2,Female,-3,IT,Analyst,25000,-1,01-03-2023,Pune,Gujarat,Not Assigned,5,97,false
3,Employee_3,Male,24,IT,Manager,45000,5,2023-01-04,Ahmedabad,Rajasthan,Rohit,5,88,false
4,Employee_4,Male,36,Sales,Manager,-15000,1,05/01/2023,Ahmedabad,Delhi,Rohit,3,88,false
5,Employee_5,Female,-3,Finance,Executive,70000,5,01-06-2023,Pune,Maharashtra,Amit,6,88,true
6,Employee_6,Male,36,Sales,Analyst,70000,1,2023-01-07,Pune,Gujarat,Not Assigned,4,92,true
7,Employee_7,Male,30,Sales,Executive,25000,5,08/01/2023,Mumbai,Delhi,Rohit,6,110,false
8,Employee_8,Female,36,Marketing,Executive,45000,3,01-09-2023,Pune,Rajasthan,Rohit,6,110,false
9,Employee_9,Female,36,Sales,Executive,-15000,5,2023-01-10,Jaipur,Gujarat,Rohit,6,92,true
10,Employee_10,Female,36,Sales,Engineer,25000,5,11/01/2023,Jaipur,Rajasthan,Not Assigned,6,92,false


#### **Insights**

##### Replaces missing values with meaningful defaults.

In [0]:
null_values = cleaned_df.select([
    count(when(col(x).isNull(), x)).alias(x)
    for x in cleaned_df.columns
])

display(null_values)

employee_id,employee_name,gender,age,department,designation,salary,experience,joining_date,city,state,manager_name,performance_rating,attendance_percentage,active
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


#### **Insights**

##### Ensures that missing values have been handled correctly.

##5.4: Handling Negative Values

In [0]:
cleaned_df = cleaned_df.withColumn("experience",when(col("experience") < 0,lit(0))
                                .otherwise(col("experience")))

In [0]:
cleaned_df = cleaned_df.withColumn("salary",when(col("salary") < 0, col("salary")*(-1))
                            .otherwise(col("salary")))

In [0]:
avg_age = cleaned_df.select(avg(col("age")).cast("int")).collect()[0][0]

cleaned_df = cleaned_df.withColumn("age",when(col("age")<0,avg_age)
                        .otherwise(col("age")))

In [0]:
cleaned_df = cleaned_df.sort(col("employee_id").asc())

In [0]:
print("After Performing Basic Cleaning")
display(cleaned_df.limit(10))

After Performing Basic Cleaning


employee_id,employee_name,gender,age,department,designation,salary,experience,joining_date,city,state,manager_name,performance_rating,attendance_percentage,active
1,Employee_1,Male,42,Finance,Engineer,45000,5,02/01/2023,Pune,Gujarat,Neha,6,110,true
2,Employee_2,Female,25,IT,Analyst,25000,0,01-03-2023,Pune,Gujarat,Not Assigned,5,97,false
3,Employee_3,Male,24,IT,Manager,45000,5,2023-01-04,Ahmedabad,Rajasthan,Rohit,5,88,false
4,Employee_4,Male,36,Sales,Manager,15000,1,05/01/2023,Ahmedabad,Delhi,Rohit,3,88,false
5,Employee_5,Female,25,Finance,Executive,70000,5,01-06-2023,Pune,Maharashtra,Amit,6,88,true
6,Employee_6,Male,36,Sales,Analyst,70000,1,2023-01-07,Pune,Gujarat,Not Assigned,4,92,true
7,Employee_7,Male,30,Sales,Executive,25000,5,08/01/2023,Mumbai,Delhi,Rohit,6,110,false
8,Employee_8,Female,36,Marketing,Executive,45000,3,01-09-2023,Pune,Rajasthan,Rohit,6,110,false
9,Employee_9,Female,36,Sales,Executive,15000,5,2023-01-10,Jaipur,Gujarat,Rohit,6,92,true
10,Employee_10,Female,36,Sales,Engineer,25000,5,11/01/2023,Jaipur,Rajasthan,Not Assigned,6,92,false


#### **Insights**

##### Handles invalid negative values in age, salary, and experience columns by replacing them with valid values to improve data quality.


## 5.5: Removing Duplicate Records

In [0]:
print('Before removing duplicates')
cleaned_df.count()

Before removing duplicates


1269

In [0]:
cleaned_df = cleaned_df.dropDuplicates()

In [0]:
print('After removing duplicates')
cleaned_df.count()

After removing duplicates


1269

#### **Insights**

##### Removes duplicate employee records from the dataset.

# Step 6: Delta Lake Processing

## 6.1: Create Delta Table 

In [0]:
cleaned_df.write.format("delta") \
  .mode("overwrite") \
  .saveAsTable("employee_table")

#### **Insights**

##### Stores the cleaned dataset as a Delta table.

In [0]:
display(spark.sql("SELECT * FROM employee_table").limit(10))

employee_id,employee_name,gender,age,department,designation,salary,experience,joining_date,city,state,manager_name,performance_rating,attendance_percentage,active
1,Employee_1,Male,42,Finance,Engineer,45000,5,02/01/2023,Pune,Gujarat,Neha,6,110,true
2,Employee_2,Female,25,IT,Analyst,25000,0,01-03-2023,Pune,Gujarat,Not Assigned,5,97,false
3,Employee_3,Male,24,IT,Manager,45000,5,2023-01-04,Ahmedabad,Rajasthan,Rohit,5,88,false
4,Employee_4,Male,36,Sales,Manager,15000,1,05/01/2023,Ahmedabad,Delhi,Rohit,3,88,false
5,Employee_5,Female,25,Finance,Executive,70000,5,01-06-2023,Pune,Maharashtra,Amit,6,88,true
6,Employee_6,Male,36,Sales,Analyst,70000,1,2023-01-07,Pune,Gujarat,Not Assigned,4,92,true
7,Employee_7,Male,30,Sales,Executive,25000,5,08/01/2023,Mumbai,Delhi,Rohit,6,110,false
8,Employee_8,Female,36,Marketing,Executive,45000,3,01-09-2023,Pune,Rajasthan,Rohit,6,110,false
9,Employee_9,Female,36,Sales,Executive,15000,5,2023-01-10,Jaipur,Gujarat,Rohit,6,92,true
10,Employee_10,Female,36,Sales,Engineer,25000,5,11/01/2023,Jaipur,Rajasthan,Not Assigned,6,92,false


#### **Insights**

##### Verifies that the Delta table is created successfully.

## 6.2: Read Incremental Dataset

In [0]:
df_increment = spark.read \
.format("csv") \
.option("header","true") \
.option("inferSchema","true") \
.load("/Volumes/dbacademy/default/myvolume/source_file.csv")

In [0]:
display(df_increment)

employee_id,employee_name,gender,age,department,designation,salary,experience,joining_date,city,state,manager_name,performance_rating,attendance_percentage,active
101,Rahul Sharma,Male,30,IT,Software Engineer,780000,5,2021-06-15,Jaipur,Rajasthan,Anita Verma,5,97,true
450,Priya Mehta,Female,28,HR,HR Executive,550000,4,2022-02-10,Delhi,Delhi,Rohit Khanna,4,95,true
890,Amit Singh,Male,35,Finance,Senior Analyst,900000,8,2018-09-20,Lucknow,Uttar Pradesh,Sneha Kapoor,5,98,true
1333,Neha Gupta,Female,29,Marketing,Marketing Executive,600000,3,2025-01-10,Mumbai,Maharashtra,Vikram Joshi,4,96,true
1334,Karan Mehta,Male,32,Sales,Sales Manager,850000,7,2024-08-05,Pune,Maharashtra,Rajesh Sharma,5,99,true


#### **Insights**

##### Loads new employee records for the upsert operation.

## 6.3: Convert Reference Of Delta Table(employee_table)

In [0]:
delta_table = DeltaTable.forName(spark,"employee_table")

#### **Insights**

##### Creates a reference to the existing Delta table for updates.

## 6.4: Merge Operation(Upsert)

In [0]:
delta_table.alias("target") \
.merge(
    df_increment.alias("source"),
    "target.employee_id = source.employee_id"
) \
.whenMatchedUpdateAll() \
.whenNotMatchedInsertAll() \
.execute()

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

#### **Insights**

##### Updates existing records and inserts new records into the Delta table.

# STEP 7: Validation

## 7.1: Row Count

In [0]:
print("Count Before Merge Operation: ")
cleaned_df.count()

Count Before Merge Operation: 


1269

In [0]:
print("Count After Merge Operation:")
spark.sql("select count(*) from employee_table").collect()[0][0]

Count After Merge Operation:


1271

#### **Insights**

##### Confirms the total number of records after the merge.

## 7.2: Duplicate Check

In [0]:
duplicate_employees = spark.sql("""select employee_id,count(*) as employee_count
          from employee_table
          group by employee_id
          having count(*) > 1""")

display(duplicate_employees)

employee_id,employee_count


#### **Insights**

##### Verifies that no duplicate employee IDs exist after the merge.

# STEP 8: Summary

In [0]:
%sql
select *
from employee_table
order by employee_id

employee_id,employee_name,gender,age,department,designation,salary,experience,joining_date,city,state,manager_name,performance_rating,attendance_percentage,active
1,Employee_1,Male,42,Finance,Engineer,45000,5,02/01/2023,Pune,Gujarat,Neha,6,110,true
2,Employee_2,Female,25,IT,Analyst,25000,0,01-03-2023,Pune,Gujarat,Not Assigned,5,97,false
3,Employee_3,Male,24,IT,Manager,45000,5,2023-01-04,Ahmedabad,Rajasthan,Rohit,5,88,false
4,Employee_4,Male,36,Sales,Manager,15000,1,05/01/2023,Ahmedabad,Delhi,Rohit,3,88,false
5,Employee_5,Female,25,Finance,Executive,70000,5,01-06-2023,Pune,Maharashtra,Amit,6,88,true
6,Employee_6,Male,36,Sales,Analyst,70000,1,2023-01-07,Pune,Gujarat,Not Assigned,4,92,true
7,Employee_7,Male,30,Sales,Executive,25000,5,08/01/2023,Mumbai,Delhi,Rohit,6,110,false
8,Employee_8,Female,36,Marketing,Executive,45000,3,01-09-2023,Pune,Rajasthan,Rohit,6,110,false
9,Employee_9,Female,36,Sales,Executive,15000,5,2023-01-10,Jaipur,Gujarat,Rohit,6,92,true
10,Employee_10,Female,36,Sales,Engineer,25000,5,11/01/2023,Jaipur,Rajasthan,Not Assigned,6,92,false


#### **Insight**

##### Merge operation successfully completed on the Delta table with the following results:

##### New Records Inserted: 1333, 1334

##### Existing Records Updated: 101, 450, 890

In [0]:
print("Target records :", cleaned_df.count())

print("Source records :", df_increment.count())

print("Final records :", spark.table("employee_table").count())

Target records : 1269
Source records : 5
Final records : 1271


#### **Insights**

##### Summarizes the source, target, and final record counts after processing.